<a href="https://colab.research.google.com/github/lbush5355/PoseAI/blob/main/PoseAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Step 1: Environment Setup & Module Synchronization

from google.colab import drive
import os
import sys
import shutil
import logging
import glob
import collections
from datetime import datetime
import requests

# Mount Google Drive (handle if already mounted)
try:
    drive.mount('/content/drive', force_remount=False)
except RuntimeError as e:
    if "already mounted" in str(e):
        print("Drive already mounted")
    else:
        raise

PROJECT_PATH = "/content/drive/MyDrive/PoseAI" # @param {type:"string"}
GIT_REPO = "https://github.com/lbush5355/PoseAI.git" # @param {type:"string"}
GIT_REF = "main" # @param {type:"string"}

FAST_LANE = "/content/fast_lane"
SRC_DIR = f"{FAST_LANE}/src"
REPO_CLONE_DIR = "/tmp/poseai_repo"

os.makedirs(SRC_DIR, exist_ok=True)

# Step A: clone repo and copy src/* into fast_lane.
# This must stay inline — we can't import from a repo we haven't cloned yet.
print(f"Cloning {GIT_REPO} @ {GIT_REF}...")
if os.path.exists(REPO_CLONE_DIR):
    shutil.rmtree(REPO_CLONE_DIR)
clone_rc = os.system(
    f"git clone -q --depth 1 --branch {GIT_REF} {GIT_REPO} {REPO_CLONE_DIR}"
)
if clone_rc != 0:
    raise RuntimeError(f"git clone failed (rc={clone_rc}): {GIT_REPO} ref={GIT_REF}")

repo_src = f"{REPO_CLONE_DIR}/src"
for fname in sorted(os.listdir(repo_src)):
    if fname.endswith(".py"):
        shutil.copy2(f"{repo_src}/{fname}", f"{SRC_DIR}/{fname}")

head_sha = os.popen(f"git -C {REPO_CLONE_DIR} rev-parse --short HEAD").read().strip()
print(f"Modules synced @ {GIT_REF} ({head_sha})")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Configure logging. force=True replaces Colab's pre-existing root
# handlers so our INFO-level logs from poseai.* actually surface; without
# it, basicConfig is a no-op and runtime.py / pipeline.py progress lines
# never reach the cell output.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-24s | %(levelname)-8s | %(message)s',
    force=True,
)
logger = logging.getLogger("poseai")

# Step B: hand off to runtime.setup_environment for everything else
# (pip deps, i386 libs, fpocket build, binary download + ELF validation).
from runtime import setup_environment

ctx = setup_environment(FAST_LANE, git_ref=GIT_REF, head_sha=head_sha)
os.environ["PATH"] += f":{ctx.bin_dir}:/usr/local/bin"

# Step C: import 3rd-party libs at cell scope so subsequent cells can use
# them as globals (Colab cells share a module namespace).
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, Draw
import py3Dmol

# Step D: import project modules and validate config.
from config import get_config
config = get_config()
config.validate_all()

import pipeline
import run_history
from preprocessor import (
    ProteinLigandPrep, fetch_rcsb_smiles, fetch_rcsb_ideal_sdf,
    fetch_rcsb_entry_ligand_codes,
    extract_ligand_code_candidates_from_mol2, get_ligand_centroid,
)
from docking import EnsembleManager, EngineType
from consensus import ConsensusAnalyzer
from visualizer import DockingVisualizer

logger.info(f"Environment ready @ {ctx.git_ref} ({ctx.head_sha})")
logger.info(f"  src/: {ctx.src_dir}")
logger.info(f"  bin/: {ctx.bin_dir}")


In [ ]:
# @title Step 2: PoseAI Pipeline Configuration

TARGET_PDB = "1owe" # @param {type:"string"}
LIGAND_CODE = "675" # @param {type:"string"}
LIGAND_SMILES = "" # @param {type:"string"}
# (Leave LIGAND_SMILES empty to dynamically fetch reference from RCSB PDB)

EXHAUSTIVENESS = 32 # @param {type:"slider", min:1, max:64}
POSES_PER_ENGINE = 20 # @param {type:"number"}
BOX_PADDING = 5 # @param {type:"slider", min:5, max:20}
RMSD_THRESHOLD = 2.5 # @param {type:"number"}
TIMEOUT_SECONDS = 3600 # @param {type:"slider", min:600, max:7200}

USE_GNINA = True # @param {type:"boolean"}
USE_SMINA = True # @param {type:"boolean"}
USE_LEDOCK = True # @param {type:"boolean"}

print("Parameters loaded from Step 2")

In [ ]:
# =============================================================================
# @title Step 3: PoseAI Single-Target Pipeline (v0.9)
# =============================================================================

# Read parameters from Cell 2 (globals); fall back to a working default.
TARGET_PDB = globals().get('TARGET_PDB', '1owe').lower()
LIGAND_CODE = globals().get('LIGAND_CODE', '') or None
LIGAND_SMILES = globals().get('LIGAND_SMILES', '') or None

RESULTS_DIR = f"{FAST_LANE}/results/{TARGET_PDB.upper()}"
DRIVE_RESULTS = f"{PROJECT_PATH}/results/{TARGET_PDB.upper()}"

print("\n" + "="*70)
print(f"POSEAI PIPELINE v0.9  |  {TARGET_PDB.upper()}")
print("="*70)
print(f"  PDB ID:       {TARGET_PDB.upper()}")
print(f"  Ligand Code:  {LIGAND_CODE or '(auto-resolve)'}")
print(f"  Engines:      {[e for e in ['GNINA','SMINA','LEDOCK'] if globals().get(f'USE_{e}', True)]}")

engines = []
if USE_GNINA: engines.append(EngineType.GNINA)
if USE_SMINA: engines.append(EngineType.SMINA)
if USE_LEDOCK: engines.append(EngineType.LEDOCK)

params = pipeline.PipelineParams(
    exhaustiveness=EXHAUSTIVENESS,
    poses_per_engine=POSES_PER_ENGINE,
    box_padding=BOX_PADDING,
    rmsd_threshold=RMSD_THRESHOLD,
    timeout=TIMEOUT_SECONDS,
    engines=engines,
)

result = pipeline.run_from_rcsb(
    target_pdb=TARGET_PDB,
    ligand_code=LIGAND_CODE,
    ligand_smiles=LIGAND_SMILES,
    work_dir=FAST_LANE,
    results_dir=RESULTS_DIR,
    params=params,
)

# Summary
print("\n" + "="*70)
print(f"  Status:      {result.status}")
print(f"  Confidence:  {result.confidence}")
if result.native_rmsd is not None:
    print(f"  Native RMSD: {result.native_rmsd:.3f} A")
print("="*70)

if not result.cluster_df.empty:
    print(f"\nConsensus clusters ({len(result.cluster_df)}):\n")
    print(result.cluster_df.to_string(index=False))

    # 3D visualization of the best cluster
    viz = DockingVisualizer(result.receptor_pdb)
    viz.create_interactive_view()
    viz.add_consensus_cluster(
        result.analyzer.all_poses,
        result.best_pose_indices.tolist(),
        label="Consensus",
    )
    viz.show()

    # Archive to Drive
    os.makedirs(RESULTS_DIR, exist_ok=True)
    result.cluster_df.to_csv(os.path.join(RESULTS_DIR, "cluster_summary.csv"), index=False)
    shutil.copytree(RESULTS_DIR, DRIVE_RESULTS, dirs_exist_ok=True)
    print(f"\nResults archived to {DRIVE_RESULTS}")
else:
    print("\nNo consensus clusters found.")

In [ ]:
# =============================================================================
# @title Step 4: Validation Overlay (Native vs Predicted)
# =============================================================================

if result.native_rmsd is None:
    print("✗ Validation skipped: no consensus pose to compare against the crystal.")
else:
    print(f"Strict In-Place Heavy-Atom RMSD: {result.native_rmsd:.3f} Å  →  {result.status}")
    if result.status == "Success":
        print("  Within the universally accepted 2.0 Å threshold (near-native pose).")
    elif result.status == "Acceptable":
        print("  Correct general binding mode; minor deviation.")
    else:
        print("  Significant deviation from native crystal pose.")

    # Native (green) + predicted (cyan) overlay
    viz_val = DockingVisualizer(result.receptor_pdb)
    viz_val.create_interactive_view()
    with open(result.ligand_mol2, 'r') as f:
        viz_val.view.addModel(f.read(), 'mol2')
        viz_val.view.setStyle({'model': -1}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.15}})
    viz_val.view.addModel(Chem.MolToMolBlock(result.top_pose_mol), 'mol')
    viz_val.view.setStyle({'model': -1}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.15}})
    viz_val.view.zoomTo()
    viz_val.show()
    print("■ Green: Native Crystal Ligand  |  ■ Cyan: Top Predicted Consensus Pose")


In [ ]:
# =============================================================================
# @title Step 5: Local Dataset Batch Validation
# =============================================================================

DATASET_DIR = f"{PROJECT_PATH}/dataset"
BATCH_RESULTS_DIR = f"{PROJECT_PATH}/batch_results"
RUN_HISTORY_CSV = f"{PROJECT_PATH}/run_history.csv"
os.makedirs(BATCH_RESULTS_DIR, exist_ok=True)

run_id = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print("="*70)
print("STAGE 5: LOCAL DATASET BATCH VALIDATION")
print(f"Run ID: {run_id}")
print("="*70)

if not os.path.exists(DATASET_DIR) or not os.listdir(DATASET_DIR):
    print(f"Dataset directory not found or empty: {DATASET_DIR}")
    print("Please ensure your target folders (e.g., \'1hsg\') are uploaded.")
else:
    target_folders = sorted(
        f for f in os.listdir(DATASET_DIR)
        if os.path.isdir(os.path.join(DATASET_DIR, f))
    )
    print(f"Found {len(target_folders)} targets in {DATASET_DIR}\n")

    engines = []
    if USE_GNINA: engines.append(EngineType.GNINA)
    if USE_SMINA: engines.append(EngineType.SMINA)
    if USE_LEDOCK: engines.append(EngineType.LEDOCK)

    params = pipeline.PipelineParams(
        exhaustiveness=EXHAUSTIVENESS,
        poses_per_engine=POSES_PER_ENGINE,
        box_padding=BOX_PADDING,
        rmsd_threshold=RMSD_THRESHOLD,
        timeout=TIMEOUT_SECONDS,
        engines=engines,
    )

    rows = []
    for target_id in target_folders:
        print(f"\nProcessing Target: {target_id.upper()}")
        print("-" * 30)
        target_path = os.path.join(DATASET_DIR, target_id)
        target_results_dir = os.path.join(BATCH_RESULTS_DIR, target_id.upper())

        try:
            result = pipeline.run_from_local(
                target_id=target_id,
                target_path=target_path,
                work_dir=FAST_LANE,
                results_dir=target_results_dir,
                params=params,
            )
        except Exception as e:
            print(f"  -> Failed: {e}")
            rows.append({
                "Target": target_id.upper(), "Status": "Error",
                "RMSD": None, "Confidence": None, "Composition": None,
            })
            continue

        if result.native_rmsd is not None:
            print(f"  -> Native RMSD: {result.native_rmsd:.3f} Å")
        print(f"  -> Status:      {result.status}")
        print(f"  -> Confidence:  {result.confidence}")

        composition_str = None
        if result.best_pose_indices is not None and result.analyzer is not None:
            engine_counts = collections.Counter(
                result.analyzer._metadata[i].get("engine", "UNKNOWN")
                for i in result.best_pose_indices
            )
            print("  -> Top Cluster Engine Breakdown:")
            for eng, count in engine_counts.items():
                print(f"       - {eng}: {count} poses")
            composition_str = ", ".join(f"{e}:{c}" for e, c in engine_counts.items())

        run_history.append_run(
            result,
            csv_path=RUN_HISTORY_CSV,
            run_id=run_id,
            exhaustiveness=EXHAUSTIVENESS,
            poses_per_engine=POSES_PER_ENGINE,
        )

        rows.append({
            "Target": result.target_id,
            "Status": result.status,
            "RMSD": result.native_rmsd,
            "Confidence": result.confidence,
            "Composition": composition_str,
        })

        # 3D overlay HTML for offline inspection
        if result.top_pose_mol is not None:
            html_path = os.path.join(target_results_dir, f"{result.target_id}_view.html")
            view = py3Dmol.view(width=800, height=600)
            view.setBackgroundColor("white")
            with open(result.receptor_pdb) as f:
                view.addModel(f.read(), "pdb")
            view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray"}})
            with open(result.ligand_mol2) as f:
                view.addModel(f.read(), "mol2")
            view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.15}})
            view.addModel(Chem.MolToMolBlock(result.top_pose_mol), "mol")
            view.setStyle({"model": 2}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.15}})
            view.zoomTo({"model": 1})
            with open(html_path, "w") as f:
                f.write(view._make_html())

    summary_df = pd.DataFrame(rows)
    summary_csv = os.path.join(BATCH_RESULTS_DIR, "batch_summary.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"\nBatch validation complete! Summary saved to: {summary_csv}\n")
    display(summary_df)

    # Per-target historical statistics
    print("\n" + "="*70)
    print("PER-TARGET HISTORICAL STATISTICS (all runs in run_history.csv)")
    print("="*70)
    stats_df = run_history.per_target_stats(RUN_HISTORY_CSV)
    if not stats_df.empty:
        display(stats_df)
    else:
        print("No history available yet.")
